In [3]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [5]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="payamvha/NorthTTS", 
                  repo_type="dataset", local_dir="./NorthTTS")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:02<00:00,  2.07it/s]


'/home/ubuntu/NorthTTS'

In [6]:
files = glob('NorthTTS/*/*.parquet')
files

['NorthTTS/data/train-00000-of-00002.parquet',
 'NorthTTS/data/train-00001-of-00002.parquet',
 'NorthTTS/data/validation-00000-of-00001.parquet']

In [8]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [9]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 712/712 [00:54<00:00, 13.00it/s]


In [10]:
len(data)

1583

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'NorthTTS_audio/NorthTTS-data-train-00000-of-00002_0.mp3',
 'text': 'دانش\u200cآموزان برای امتحان فردا آماده می\u200cشوند.',
 'speaker': 'NorthTTS_audio'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'NorthTTS_audio')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 624.34ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  149kB /  149kB, 17.7kB/s  
Processing Files (1 / 1): 100%|██████████|  149kB /  149kB, 17.7kB/s  
New Data Upload: 100%|██████████|  149kB /  149kB, 17.7kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:08<00:00,  8.71s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/f490658b6a2fa105648584c2f41199df0a54101e', commit_message='Upload dataset', commit_description='', oid='f490658b6a2fa105648584c2f41199df0a54101e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('NorthTTS_audio-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
!zip -rq NorthTTS_audio.zip NorthTTS_audio

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


In [16]:
# !hf upload malaysia-ai/Multilingual-TTS NorthTTS_audio.zip --repo-type=dataset

In [17]:
# !zip -rq NorthTTS_audio_neucodec.zip NorthTTS_audio_neucodec

In [19]:
# !hf upload malaysia-ai/Multilingual-TTS NorthTTS_audio_neucodec.zip --repo-type=dataset